# Exploratory Data Analysis - Stock Market Trend Analysis

This notebook performs comprehensive EDA on NSE/BSE equity data with technical indicators.

**Dataset:** 50 Nifty 50 constituent stocks (Jan 2025 - Apr 2025)  
**Features:** OHLCV, RSI, MACD, Bollinger Bands, Moving Averages, Volatility, Trading Signals

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/nse_bse_features.csv')
df['date'] = pd.to_datetime(df['date'])

print('='*70)
print('DATASET OVERVIEW')
print('='*70)
print(f'Shape: {df.shape}')
print(f'Date Range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Unique Stocks: {df["symbol"].nunique()}')
print(f'Sectors: {df["sector"].nunique()}')
print(f'Columns: {list(df.columns)}')
print('='*70)

In [ ]:
# Display first 10 rows
df.head(10)

In [ ]:
# Dataset info
df.info()

## 2. Missing Value Analysis

In [ ]:
# Missing values heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Records (rows)', fontsize=12)
plt.tight_layout()
plt.show()

# Missing value summary
missing_summary = df.isnull().sum()
missing_pct = (missing_summary / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_summary, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print('\nMissing Value Summary:')
print(missing_df)

## 3. RSI Distribution by Sector

In [ ]:
# RSI distribution histogram by sector
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

sectors = df['sector'].unique()

for idx, sector in enumerate(sectors):
    if idx < len(axes):
        sector_data = df[df['sector'] == sector]['rsi_14'].dropna()
        axes[idx].hist(sector_data, bins=30, edgecolor='black', alpha=0.7, color=f'C{idx}')
        axes[idx].axvline(x=30, color='green', linestyle='--', linewidth=2, label='Oversold (30)')
        axes[idx].axvline(x=70, color='red', linestyle='--', linewidth=2, label='Overbought (70)')
        axes[idx].set_title(f'{sector} Sector', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('RSI (14-day)')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend(fontsize=8)

plt.suptitle('RSI Distribution by Sector', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. MACD Signal Crossover for INFY (Jan-Apr 2025)

In [ ]:
# Filter INFY data
infy_df = df[df['symbol'] == 'INFY'].sort_values('date').copy()

fig, ax1 = plt.subplots(figsize=(14, 7))

# Price plot
ax1.plot(infy_df['date'], infy_df['close'], 'b-', linewidth=2, label='Close Price')
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Close Price (₹)', color='b', fontsize=12)
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_title('INFY - Price and MACD Crossover Analysis (Jan-Apr 2025)', fontsize=16, fontweight='bold')

# MACD plot on secondary axis
ax2 = ax1.twinx()
ax2.plot(infy_df['date'], infy_df['macd'], 'g-', linewidth=1.5, label='MACD Line')
ax2.plot(infy_df['date'], infy_df['macd_signal'], 'r-', linewidth=1.5, label='Signal Line')
ax2.set_ylabel('MACD Value', color='g', fontsize=12)
ax2.tick_params(axis='y', labelcolor='g')

# Mark crossover points
bullish = infy_df[infy_df['signal'] == 'BUY']
bearish = infy_df[infy_df['signal'] == 'SELL']

ax1.scatter(bullish['date'], bullish['close'], marker='^', color='green', s=100, 
           label='BUY Signal', zorder=5, edgecolors='black', linewidth=1)
ax1.scatter(bearish['date'], bearish['close'], marker='v', color='red', s=100, 
           label='SELL Signal', zorder=5, edgecolors='black', linewidth=1)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Correlation Heatmap of Technical Indicators

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['rsi_14', 'macd', 'macd_hist', 'sma_20', 'sma_50', 
                'daily_return', 'volatility_20d', 'vol_zscore', 'close', 'volume']

# Calculate correlation matrix
corr_matrix = df[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap - Technical Indicators', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 6. Sector-wise Box Plot of Daily Returns

In [ ]:
# Box plot of daily returns by sector
plt.figure(figsize=(14, 7))
sns.boxplot(data=df, x='sector', y='daily_return', palette='husl')
plt.title('Sector-wise Daily Returns Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Sector', fontsize=12)
plt.ylabel('Daily Return', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
plt.tight_layout()
plt.show()

# Statistics table
print('\nDaily Returns Statistics by Sector:')
print(df.groupby('sector')['daily_return'].agg(['mean', 'std', 'min', 'max', 'count']).round(6))

## 7. Volume vs Volatility Scatter Plot

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(12, 7))

# Sample data for better visualization (10% random sample)
sample_df = df.sample(frac=0.1, random_state=42)

scatter = sns.scatterplot(data=sample_df, x='volatility_20d', y='volume', 
                         hue='sector', size='close', sizes=(20, 200),
                         alpha=0.7, palette='husl')

# Add regression line
sns.regplot(data=sample_df, x='volatility_20d', y='volume', 
           scatter=False, color='black', line_kws={'linewidth': 2})

plt.title('Volume vs Volatility (Colored by Sector, Sized by Close Price)', 
         fontsize=16, fontweight='bold')
plt.xlabel('Volatility (20-day Annualized)', fontsize=12)
plt.ylabel('Volume', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Signal Distribution Pie Chart

In [ ]:
# Signal distribution
signal_counts = df['signal'].value_counts()

plt.figure(figsize=(10, 8))
colors = {'BUY': '#00B050', 'SELL': '#FF0000', 'HOLD': '#A5A5A5'}
wedges, texts, autotexts = plt.pie(signal_counts.values, labels=signal_counts.index, 
                                    autopct='%1.1f%%', colors=[colors[s] for s in signal_counts.index],
                                    startangle=90, explode=(0.05, 0.05, 0.05),
                                    textprops={'fontsize': 14, 'fontweight': 'bold'})

# Style autotexts
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

plt.title('Trading Signal Distribution', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('\nSignal Counts:')
print(signal_counts)

## 9. Bollinger Band Chart for HDFCBANK

In [ ]:
# Filter HDFCBANK data
hdfc_df = df[df['symbol'] == 'HDFCBANK'].sort_values('date').copy()

fig, ax = plt.subplots(figsize=(14, 7))

# Plot close price
ax.plot(hdfc_df['date'], hdfc_df['close'], 'b-', linewidth=2, label='Close Price')

# Plot Bollinger Bands
ax.plot(hdfc_df['date'], hdfc_df['bb_upper'], 'r--', linewidth=1.5, label='Upper Band (SMA + 2σ)')
ax.plot(hdfc_df['date'], hdfc_df['bb_lower'], 'g--', linewidth=1.5, label='Lower Band (SMA - 2σ)')
ax.plot(hdfc_df['date'], hdfc_df['bb_mid'], 'orange', linewidth=1.5, label='Middle Band (SMA 20)')

# Fill between bands
ax.fill_between(hdfc_df['date'], hdfc_df['bb_upper'], hdfc_df['bb_lower'], 
                alpha=0.15, color='blue', label='Bollinger Band Width')

ax.set_title('HDFCBANK - Bollinger Bands (Jan-Apr 2025)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (₹)', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary Statistics Table

In [ ]:
# Comprehensive summary statistics by sector
summary_stats = df.groupby('sector').agg({
    'daily_return': ['mean', 'std', 'min', 'max'],
    'rsi_14': 'mean',
    'volume': 'mean',
    'close': 'mean',
    'volatility_20d': 'mean',
    'signal': lambda x: x.value_counts().index[0]  # Most common signal
}).round(6)

summary_stats.columns = ['Avg Daily Return', 'Return Std Dev', 'Min Return', 'Max Return',
                         'Avg RSI', 'Avg Volume', 'Avg Close', 'Avg Volatility', 'Dominant Signal']

print('='*100)
print('SECTOR-WISE SUMMARY STATISTICS')
print('='*100)
print(summary_stats)
print('='*100)

In [ ]:
# Overall dataset summary
print('\n' + '='*70)
print('OVERALL DATASET SUMMARY')
print('='*70)
print(f'Total Records: {len(df):,}')
print(f'Time Period: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Trading Days: {df["date"].nunique()}')
print(f'Stocks Analyzed: {df["symbol"].nunique()}')
print(f'Sectors Covered: {df["sector"].nunique()}')
print(f'\nAverage Daily Return: {df["daily_return"].mean()*100:.4f}%')
print(f'Average Volatility: {df["volatility_20d"].mean()*100:.4f}%')
print(f'Average RSI: {df["rsi_14"].mean():.2f}')
print(f'\nSignal Distribution:')
for signal, count in df['signal'].value_counts().items():
    pct = (count / len(df)) * 100
    print(f'  {signal}: {count:,} ({pct:.1f}%)')
print('='*70)